# 01 · Bronze · Olist (canal marketplace)
Ingere os CSVs do dataset público Olist, que representa o canal **marketplace** da Lojas Horizonte.

- **Origem:** `/Volumes/horizonte/bronze/landing/olist/`
- **Fonte:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), licença CC BY-NC-SA 4.0
- **Tabelas geradas:** `olist_orders`, `olist_order_items`, `olist_products`, `olist_customers`, `olist_category_translation`, `olist_order_reviews`

### Ingestão
Cada CSV vira uma tabela bronze, sem nenhuma transformação:
- **Tudo como texto** (`inferSchema => false`): a bronze guarda o dado exatamente como chegou. A tipagem acontece na silver.
- **Colunas de auditoria:** `_arquivo_origem` e `_ingerido_em` registram de onde e quando cada linha veio.
- **Carga completa** (`CREATE OR REPLACE`): adequada a um dataset estático.

**Equivalente no BigQuery:** os CSVs ficam num bucket do Cloud Storage, lidos por uma tabela externa com colunas `STRING`. A coluna de origem vem de `_FILE_NAME` e a data de carga de `CURRENT_TIMESTAMP()`. A estrutura da consulta é a mesma.

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.olist_orders AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/olist/olist_orders_dataset.csv',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.olist_order_items AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/olist/olist_order_items_dataset.csv',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.olist_products AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/olist/olist_products_dataset.csv',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.olist_customers AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/olist/olist_customers_dataset.csv',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.olist_category_translation AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/olist/product_category_name_translation.csv',
                format => 'csv', header => true, inferSchema => false);

### Avaliações
A tabela `olist_order_reviews` tem comentários com quebras de linha dentro do texto e aspas escapadas por duplicação (`""`). Por isso a leitura usa `multiLine => true` e `escape => '"'`; sem elas, as linhas se partem e as colunas se deslocam.

**No BigQuery:** tabela externa com `allow_quoted_newlines = true`.

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.olist_order_reviews AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/olist/olist_order_reviews_dataset.csv',
                format => 'csv', header => true, inferSchema => false,
                multiLine => true, escape => '"');

In [0]:
SELECT 'olist_orders' AS tabela, COUNT(*) AS linhas FROM horizonte.bronze.olist_orders            -- 99.441
UNION ALL SELECT 'olist_order_items', COUNT(*) FROM horizonte.bronze.olist_order_items             -- 112.650
UNION ALL SELECT 'olist_products', COUNT(*) FROM horizonte.bronze.olist_products                   -- 32.951
UNION ALL SELECT 'olist_customers', COUNT(*) FROM horizonte.bronze.olist_customers                 -- 99.441
UNION ALL SELECT 'olist_category_translation', COUNT(*) FROM horizonte.bronze.olist_category_translation -- 71
UNION ALL SELECT 'olist_order_reviews', COUNT(*) FROM horizonte.bronze.olist_order_reviews -- 99.224